# Super Mario Bros - DQN Training (Google Colab)

This notebook trains a DQN agent to play Super Mario Bros World 1-1.

**Instructions:**
1. Go to Runtime → Change runtime type → Select **GPU** (T4 is fine)
2. Run all cells in order
3. Training checkpoints and logs are saved to Google Drive

**Deadline reminder:** April 12th - get results + graphs ready for presentation

In [ ]:
# Step 1: Install dependencies
!pip install gym==0.26.2 gym-super-mario-bros==7.4.0 nes-py==8.2.1 torch numpy opencv-python matplotlib

In [ ]:
# Step 2: Mount Google Drive for saving checkpoints
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/mario-rl-agent'
os.makedirs(f'{SAVE_DIR}/checkpoints/dqn', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/logs/dqn', exist_ok=True)
print(f'Saving to: {SAVE_DIR}')

In [ ]:
# Step 3: Verify GPU is available
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime -> Change runtime type -> GPU')

In [ ]:
# Step 4: Environment Wrappers
import gym
import numpy as np
from collections import deque
import cv2


class SkipFrame(gym.Wrapper):
    """Return only every `skip`-th frame."""
    def __init__(self, env, skip=4):
        super().__init__(env)
        self._skip = skip

    def step(self, action):
        total_reward = 0.0
        for _ in range(self._skip):
            obs, reward, done, info = self.env.step(action)
            total_reward += reward
            if done:
                break
        return obs, total_reward, done, info


class GrayScaleObservation(gym.ObservationWrapper):
    """Convert RGB frames to grayscale."""
    def __init__(self, env):
        super().__init__(env)
        obs_shape = self.observation_space.shape[:2]
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=obs_shape, dtype=np.uint8
        )

    def observation(self, observation):
        return cv2.cvtColor(observation, cv2.COLOR_RGB2GRAY)


class ResizeObservation(gym.ObservationWrapper):
    """Resize frames to 84x84."""
    def __init__(self, env, shape=84):
        super().__init__(env)
        if isinstance(shape, int):
            self.shape = (shape, shape)
        else:
            self.shape = tuple(shape)
        obs_shape = self.shape + self.observation_space.shape[2:]
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=obs_shape, dtype=np.uint8
        )

    def observation(self, observation):
        return cv2.resize(observation, self.shape, interpolation=cv2.INTER_AREA)


class NormalizeObservation(gym.ObservationWrapper):
    """Normalize pixel values from [0, 255] to [0.0, 1.0]."""
    def __init__(self, env):
        super().__init__(env)
        obs_shape = self.observation_space.shape
        self.observation_space = gym.spaces.Box(
            low=0.0, high=1.0, shape=obs_shape, dtype=np.float32
        )

    def observation(self, observation):
        return np.array(observation, dtype=np.float32) / 255.0


class FrameStack(gym.Wrapper):
    """Stack the last `num_stack` frames."""
    def __init__(self, env, num_stack=4):
        super().__init__(env)
        self._num_stack = num_stack
        self._frames = deque(maxlen=num_stack)
        low = np.repeat(self.observation_space.low[np.newaxis, ...], num_stack, axis=0)
        high = np.repeat(self.observation_space.high[np.newaxis, ...], num_stack, axis=0)
        self.observation_space = gym.spaces.Box(
            low=low, high=high, dtype=self.observation_space.dtype
        )

    def reset(self):
        obs = self.env.reset()
        for _ in range(self._num_stack):
            self._frames.append(obs)
        return self._get_obs()

    def step(self, action):
        obs, reward, done, info = self.env.step(action)
        self._frames.append(obs)
        return self._get_obs(), reward, done, info

    def _get_obs(self):
        return np.array(self._frames)


class CustomRewardWrapper(gym.Wrapper):
    """Custom reward shaping for Mario."""
    def __init__(self, env):
        super().__init__(env)
        self._last_x_pos = 0
        self._last_status = 'small'

    def reset(self):
        obs = self.env.reset()
        self._last_x_pos = 0
        self._last_status = 'small'
        return obs

    def step(self, action):
        obs, reward, done, info = self.env.step(action)
        x_pos = info.get('x_pos', 0)
        x_reward = (x_pos - self._last_x_pos) / 10.0
        self._last_x_pos = x_pos

        death_penalty = -50.0 if (done and not info.get('flag_get', False)) else 0
        flag_bonus = 500.0 if info.get('flag_get', False) else 0

        status_penalty = 0
        current_status = info.get('status', 'small')
        if self._last_status in ('tall', 'fireball') and current_status == 'small':
            status_penalty = -25.0
        self._last_status = current_status

        time_penalty = -0.1
        shaped_reward = x_reward + death_penalty + flag_bonus + status_penalty + time_penalty
        return obs, shaped_reward, done, info


def make_mario_env(use_custom_rewards=True):
    """Create and wrap the Super Mario Bros environment."""
    import gym_super_mario_bros
    from nes_py.wrappers import JoypadSpace
    from gym_super_mario_bros.actions import SIMPLE_MOVEMENT

    env = gym_super_mario_bros.make('SuperMarioBros-1-1-v0')
    env = JoypadSpace(env, SIMPLE_MOVEMENT)
    if use_custom_rewards:
        env = CustomRewardWrapper(env)
    env = SkipFrame(env, skip=4)
    env = GrayScaleObservation(env)
    env = ResizeObservation(env, shape=84)
    env = NormalizeObservation(env)
    env = FrameStack(env, num_stack=4)
    return env

print('Wrappers defined successfully!')

In [ ]:
# Step 5: DQN Network, Replay Buffer, and Agent
import random
import time
import torch.nn as nn
import torch.optim as optim


class DQNetwork(nn.Module):
    """CNN: stacked frames -> Q-values per action."""
    def __init__(self, input_channels, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Linear(3136, 512), nn.ReLU(),
            nn.Linear(512, n_actions),
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


class ReplayBuffer:
    def __init__(self, capacity=100_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states), np.array(actions),
            np.array(rewards, dtype=np.float32), np.array(next_states),
            np.array(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)


class DQNAgent:
    def __init__(self, state_shape, n_actions, lr=2.5e-4, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.02, epsilon_decay=100_000,
                 buffer_size=100_000, batch_size=32, target_update_freq=10_000,
                 device='auto'):
        if device == 'auto':
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = torch.device(device)

        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.steps_done = 0

        input_channels = state_shape[0]
        self.policy_net = DQNetwork(input_channels, n_actions).to(self.device)
        self.target_net = DQNetwork(input_channels, n_actions).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.loss_fn = nn.SmoothL1Loss()
        self.replay_buffer = ReplayBuffer(buffer_size)

    @property
    def epsilon(self):
        return self.epsilon_end + (self.epsilon_start - self.epsilon_end) * max(
            0, 1 - self.steps_done / self.epsilon_decay
        )

    def select_action(self, state):
        self.steps_done += 1
        if random.random() < self.epsilon:
            return random.randrange(self.n_actions)
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            return self.policy_net(state_t).argmax(dim=1).item()

    def learn(self):
        if len(self.replay_buffer) < self.batch_size:
            return None

        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        states_t = torch.FloatTensor(states).to(self.device)
        actions_t = torch.LongTensor(actions).to(self.device)
        rewards_t = torch.FloatTensor(rewards).to(self.device)
        next_states_t = torch.FloatTensor(next_states).to(self.device)
        dones_t = torch.FloatTensor(dones).to(self.device)

        q_values = self.policy_net(states_t).gather(1, actions_t.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q_values = self.target_net(next_states_t).max(dim=1)[0]
            target_q_values = rewards_t + self.gamma * next_q_values * (1 - dones_t)

        loss = self.loss_fn(q_values, target_q_values)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), max_norm=1.0)
        self.optimizer.step()

        if self.steps_done % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        return loss.item()

    def save(self, path):
        torch.save({
            'policy_net': self.policy_net.state_dict(),
            'target_net': self.target_net.state_dict(),
            'optimizer': self.optimizer.state_dict(),
            'steps_done': self.steps_done,
        }, path)

    def load(self, path):
        checkpoint = torch.load(path, map_location=self.device)
        self.policy_net.load_state_dict(checkpoint['policy_net'])
        self.target_net.load_state_dict(checkpoint['target_net'])
        self.optimizer.load_state_dict(checkpoint['optimizer'])
        self.steps_done = checkpoint['steps_done']

print('DQN Agent defined successfully!')

In [ ]:
# Step 6: Test environment setup (quick sanity check)
env = make_mario_env(use_custom_rewards=True)
print(f'State shape: {env.observation_space.shape}')
print(f'Number of actions: {env.action_space.n}')

state = env.reset()
print(f'Initial state shape: {state.shape}')
print(f'State dtype: {state.dtype}')
print(f'State range: [{state.min():.2f}, {state.max():.2f}]')

# Run a few random steps
for i in range(5):
    action = env.action_space.sample()
    state, reward, done, info = env.step(action)
    print(f'Step {i}: reward={reward:.2f}, x_pos={info.get("x_pos", 0)}, done={done}')

env.close()
print('\nEnvironment working correctly!')

In [ ]:
# Step 7: TRAINING CONFIGURATION
# Adjust these as needed

NUM_EPISODES = 5000       # Total training episodes
BATCH_SIZE = 32           # Replay buffer batch size
LEARNING_RATE = 2.5e-4    # Adam learning rate
GAMMA = 0.99              # Discount factor
EPSILON_START = 1.0       # Starting exploration rate
EPSILON_END = 0.02        # Final exploration rate
EPSILON_DECAY = 100_000   # Steps over which epsilon decays
BUFFER_SIZE = 100_000     # Replay buffer capacity
TARGET_UPDATE = 10_000    # Steps between target network updates
LOG_INTERVAL = 20         # Episodes between progress prints
SAVE_INTERVAL = 500       # Episodes between checkpoint saves

CHECKPOINT_DIR = f'{SAVE_DIR}/checkpoints/dqn'
LOG_DIR = f'{SAVE_DIR}/logs/dqn'

print('Training configuration set!')
print(f'Episodes: {NUM_EPISODES}')
print(f'Checkpoints: {CHECKPOINT_DIR}')
print(f'Logs: {LOG_DIR}')

In [ ]:
# Step 8: RUN DQN TRAINING
# This is the main training loop - takes several hours on a T4 GPU

from pathlib import Path

# Create environment and agent
env = make_mario_env(use_custom_rewards=True)
state_shape = env.observation_space.shape
n_actions = env.action_space.n

agent = DQNAgent(
    state_shape=state_shape,
    n_actions=n_actions,
    lr=LEARNING_RATE,
    gamma=GAMMA,
    epsilon_start=EPSILON_START,
    epsilon_end=EPSILON_END,
    epsilon_decay=EPSILON_DECAY,
    buffer_size=BUFFER_SIZE,
    batch_size=BATCH_SIZE,
    target_update_freq=TARGET_UPDATE,
)

print(f'Device: {agent.device}')
print(f'Policy net parameters: {sum(p.numel() for p in agent.policy_net.parameters()):,}')
print(f'Starting training for {NUM_EPISODES} episodes...\n')
print('-' * 80)

# OPTIONAL: Resume from checkpoint
# agent.load(f'{CHECKPOINT_DIR}/best_model.pt')
# print('Resumed from checkpoint!')

# Training metrics
episode_rewards = []
episode_lengths = []
episode_x_positions = []
best_reward = float('-inf')
best_x_pos = 0
flags_reached = 0

log_file = f'{LOG_DIR}/training_log.csv'
with open(log_file, 'w') as f:
    f.write('episode,reward,length,x_pos,epsilon,loss,flag_reached,time\n')

start_time = time.time()

for episode in range(1, NUM_EPISODES + 1):
    state = env.reset()
    episode_reward = 0
    episode_length = 0
    episode_loss = 0
    loss_count = 0
    max_x_pos = 0

    while True:
        action = agent.select_action(state)
        next_state, reward, done, info = env.step(action)
        agent.replay_buffer.push(state, action, reward, next_state, done)

        loss = agent.learn()
        if loss is not None:
            episode_loss += loss
            loss_count += 1

        state = next_state
        episode_reward += reward
        episode_length += 1
        max_x_pos = max(max_x_pos, info.get('x_pos', 0))

        if done:
            break

    flag_reached = info.get('flag_get', False)
    if flag_reached:
        flags_reached += 1

    episode_rewards.append(episode_reward)
    episode_lengths.append(episode_length)
    episode_x_positions.append(max_x_pos)
    avg_loss = episode_loss / max(loss_count, 1)

    if episode_reward > best_reward:
        best_reward = episode_reward
        agent.save(f'{CHECKPOINT_DIR}/best_model.pt')

    if max_x_pos > best_x_pos:
        best_x_pos = max_x_pos

    elapsed = time.time() - start_time
    with open(log_file, 'a') as f:
        f.write(f'{episode},{episode_reward:.2f},{episode_length},{max_x_pos},'
                f'{agent.epsilon:.4f},{avg_loss:.6f},{int(flag_reached)},{elapsed:.1f}\n')

    if episode % LOG_INTERVAL == 0:
        avg_reward = np.mean(episode_rewards[-LOG_INTERVAL:])
        avg_length = np.mean(episode_lengths[-LOG_INTERVAL:])
        avg_x = np.mean(episode_x_positions[-LOG_INTERVAL:])
        print(
            f'Ep {episode:5d} | '
            f'Avg Reward: {avg_reward:8.2f} | '
            f'Avg X: {avg_x:6.0f} | '
            f'Best X: {best_x_pos:5d} | '
            f'Eps: {agent.epsilon:.3f} | '
            f'Flags: {flags_reached} | '
            f'Buffer: {len(agent.replay_buffer):6d} | '
            f'Time: {elapsed/60:.1f}m'
        )

    if episode % SAVE_INTERVAL == 0:
        agent.save(f'{CHECKPOINT_DIR}/checkpoint_ep{episode}.pt')
        print(f'  >> Checkpoint saved at episode {episode}')

# Save final model
agent.save(f'{CHECKPOINT_DIR}/final_model.pt')
env.close()

total_time = time.time() - start_time
print('\n' + '=' * 80)
print('TRAINING COMPLETE!')
print(f'Total time: {total_time / 60:.1f} minutes ({total_time / 3600:.1f} hours)')
print(f'Best reward: {best_reward:.2f}')
print(f'Best x_pos: {best_x_pos}')
print(f'Flags reached: {flags_reached} / {NUM_EPISODES}')
print(f'Models saved to: {CHECKPOINT_DIR}')
print(f'Logs saved to: {log_file}')

In [ ]:
# Step 9: Plot Training Results
# Run this during or after training to visualize progress

import matplotlib.pyplot as plt
import pandas as pd

# Load training log
df = pd.read_csv(f'{LOG_DIR}/training_log.csv')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('DQN Training Progress - Super Mario Bros World 1-1', fontsize=14, fontweight='bold')

# Smoothing window
window = 50

# 1. Episode Rewards
ax = axes[0, 0]
ax.plot(df['episode'], df['reward'], alpha=0.3, color='blue', label='Raw')
ax.plot(df['episode'], df['reward'].rolling(window).mean(), color='blue', linewidth=2, label=f'{window}-ep avg')
ax.set_xlabel('Episode')
ax.set_ylabel('Episode Reward')
ax.set_title('Reward over Training')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. X-Position (how far Mario gets)
ax = axes[0, 1]
ax.plot(df['episode'], df['x_pos'], alpha=0.3, color='green', label='Raw')
ax.plot(df['episode'], df['x_pos'].rolling(window).mean(), color='green', linewidth=2, label=f'{window}-ep avg')
ax.set_xlabel('Episode')
ax.set_ylabel('Max X Position')
ax.set_title('Mario\'s Progress (X-Position)')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Epsilon Decay
ax = axes[1, 0]
ax.plot(df['episode'], df['epsilon'], color='red', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Epsilon')
ax.set_title('Exploration Rate (Epsilon)')
ax.grid(True, alpha=0.3)

# 4. Cumulative Flags Reached
ax = axes[1, 1]
cumulative_flags = df['flag_reached'].cumsum()
ax.plot(df['episode'], cumulative_flags, color='purple', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Cumulative Flags')
ax.set_title('Level Completions')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/dqn_training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved to {SAVE_DIR}/dqn_training_results.png')

In [ ]:
# Step 10: Evaluate trained agent and record a video
# (Run after training completes)

from IPython.display import HTML
from matplotlib import animation

def record_agent(agent, num_episodes=3, max_steps=2000):
    """Record the agent playing and return frames for visualization."""
    import gym_super_mario_bros
    from nes_py.wrappers import JoypadSpace
    from gym_super_mario_bros.actions import SIMPLE_MOVEMENT

    # Use raw env for recording (not preprocessed)
    raw_env = gym_super_mario_bros.make('SuperMarioBros-1-1-v0')
    raw_env = JoypadSpace(raw_env, SIMPLE_MOVEMENT)

    # Also create wrapped env for agent decisions
    wrapped_env = make_mario_env(use_custom_rewards=False)

    all_frames = []
    results = []

    for ep in range(num_episodes):
        raw_state = raw_env.reset()
        wrapped_state = wrapped_env.reset()
        frames = [raw_state]
        total_reward = 0
        steps = 0

        for _ in range(max_steps):
            with torch.no_grad():
                state_t = torch.FloatTensor(wrapped_state).unsqueeze(0).to(agent.device)
                action = agent.policy_net(state_t).argmax(dim=1).item()

            raw_state, _, raw_done, raw_info = raw_env.step(action)
            wrapped_state, reward, done, info = wrapped_env.step(action)
            frames.append(raw_state)
            total_reward += reward
            steps += 1

            if done or raw_done:
                break

        flag = info.get('flag_get', False)
        x_pos = info.get('x_pos', 0)
        results.append({'episode': ep+1, 'reward': total_reward, 'steps': steps,
                        'x_pos': x_pos, 'flag': flag})
        print(f'Episode {ep+1}: reward={total_reward:.1f}, x_pos={x_pos}, '
              f'steps={steps}, flag={"YES" if flag else "no"}')
        all_frames.append(frames)

    raw_env.close()
    wrapped_env.close()
    return all_frames, results


def show_gameplay(frames, speed=30):
    """Animate gameplay frames in the notebook."""
    fig, ax = plt.subplots(figsize=(6, 5.5))
    ax.axis('off')
    img = ax.imshow(frames[0])

    def update(frame_idx):
        img.set_data(frames[frame_idx])
        return [img]

    anim = animation.FuncAnimation(fig, update, frames=len(frames),
                                    interval=1000/speed, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


# Load best model and evaluate
eval_agent = DQNAgent(state_shape=(4, 84, 84), n_actions=7)
eval_agent.load(f'{CHECKPOINT_DIR}/best_model.pt')
eval_agent.epsilon_start = 0.0
eval_agent.epsilon_end = 0.0
eval_agent.steps_done = eval_agent.epsilon_decay + 1

print('Recording agent gameplay...')
all_frames, results = record_agent(eval_agent, num_episodes=3)

# Show best episode
best_ep = max(range(len(results)), key=lambda i: results[i]['x_pos'])
print(f'\nShowing best episode (Episode {best_ep+1}):')
show_gameplay(all_frames[best_ep])

In [ ]:
# Step 11: Save a GIF of the agent playing (for presentation)
from PIL import Image

def save_gameplay_gif(frames, output_path, fps=30):
    """Save frames as an animated GIF."""
    images = [Image.fromarray(f) for f in frames[::2]]  # Skip every other frame for smaller file
    images[0].save(
        output_path,
        save_all=True,
        append_images=images[1:],
        duration=int(2000/fps),
        loop=0
    )
    print(f'Saved GIF to {output_path} ({len(images)} frames)')

# Save best episode as GIF
best_ep = max(range(len(results)), key=lambda i: results[i]['x_pos'])
save_gameplay_gif(all_frames[best_ep], f'{SAVE_DIR}/dqn_gameplay.gif')
print('GIF ready for presentation!')